In [ ]:
# Kaggle setup (does nothing outside Kaggle). Needs: GPU on, Internet on, competition data as input.
# The repo stays private until the deadline, so it is cloned with a GitHub token saved as a
# Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets; fine-grained token, read-only, this repo only).
import os, subprocess
REPO = "polluvl/hw2-video-detection"   # <- set once, e.g. "polluvl/hw2-video-detection"
if os.path.isdir("/kaggle/working") and not os.path.isdir("/kaggle/working/repo"):
    try:
        from kaggle_secrets import UserSecretsClient
        auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
    except Exception:
        auth = ""  # no secret -> works once the repo is public
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
                       capture_output=True)
    if r.returncode != 0:  # do not print the command: it contains the token
        raise RuntimeError("git clone failed: check REPO and the GITHUB_TOKEN secret")
    # remove the token from the saved clone (outputs of /kaggle/working can be shared)
    subprocess.run(["git", "-C", "/kaggle/working/repo", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
if os.path.isdir("/kaggle/working/repo/notebooks"):
    os.chdir("/kaggle/working/repo/notebooks")

# 05 · Query-based: RT-DETR-L (Ultralytics, COCO-pretrained)
RT-DETR predicts a fixed set of `num_queries` boxes. During training each GT box is matched to
exactly one query by the Hungarian algorithm (cost = class + L1 + GIoU); all other queries are
trained to predict "no object". Because duplicates are penalised by the matching itself, no NMS
is needed at inference. Consequence: a frame can never have more detections than queries
(default 300), while the EDA shows some frames have more objects than that.

Same budget as the CNN baseline `yolo_640` from 04: same frames, same image size, same epochs.
Requires 04 to have exported the dataset.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import RTDETR

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
from analysis import error_breakdown, recall_by_size
from common import (SMOKE, evaluate, find_data_root, list_frames, load_preds,
                    log_result, read_labels, save_preds, seed_everything, work_dir)
from yolo_utils import predict_ultralytics, read_curves

DATA = find_data_root()
OUT = work_dir()
seed_everything(42)
TRAIN_STEP, VAL_STEP = (1, 1) if SMOKE else (10, 5)
EPOCHS = 1 if SMOKE else 30
DEVICE = 0 if torch.cuda.is_available() else "cpu"
val_frames = list_frames(DATA / "val", step=VAL_STEP)
DATA_YAML = str(OUT / f"yolo_ds_step{TRAIN_STEP}" / "data.yaml")

## A config with more queries
The number of queries is an argument of the `RTDETRDecoder` head in the model yaml. Queries are
selected top-k from encoder features (not learned embeddings), so the pretrained weights still
fit when the number changes.

In [ ]:
base_yaml = Path(ultralytics.__file__).parent / "cfg/models/rt-detr/rtdetr-l.yaml"
cfg = yaml.safe_load(base_yaml.read_text())
for layer in cfg["head"]:
    if layer[2] == "RTDETRDecoder":
        layer[3] = ["nc", 256, 500]  # [nc, hidden_dim, num_queries]
Q500_YAML = OUT / "rtdetr-l-q500.yaml"
Q500_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))

## Experiments
- `rtdetr_640`      – baseline: 300 queries, 640 px, AdamW lr 1e-4 (RT-DETR recipe)
- `rtdetr_640_q500` – 500 queries (scene density: see 01_eda objects per frame)
- `rtdetr_640_lr`   – lr 2e-4 with a longer warmup (transformers are sensitive to lr; checks
  whether the baseline is limited by slow convergence within the budget)

In [ ]:
COMMON_ARGS = dict(data=DATA_YAML, epochs=EPOCHS, batch=2 if SMOKE else 8, workers=2, seed=42,
                   deterministic=False,  # grid_sample in deformable attention has no deterministic CUDA kernel
                   device=DEVICE, project=str(OUT / "ultralytics"), exist_ok=True, patience=100,
                   plots=not SMOKE, imgsz=320 if SMOKE else 640, optimizer="AdamW", lr0=1e-4,
                   cos_lr=True, close_mosaic=5)

RUNS = {
    "rtdetr_640": dict(model="rtdetr-l.pt", max_det=300),
    "rtdetr_640_q500": dict(model=str(Q500_YAML), weights="rtdetr-l.pt", max_det=500),
    "rtdetr_640_lr": dict(model="rtdetr-l.pt", max_det=300, lr0=2e-4, warmup_epochs=5),
}
if SMOKE:
    RUNS = {k: {**v, "model": v["model"].replace("rtdetr-l.pt", "rtdetr-l.yaml")} for k, v in RUNS.items()}
    for v in RUNS.values():
        v.pop("weights", None)

for name, spec in RUNS.items():
    spec = dict(spec)
    model = RTDETR(spec.pop("model"))
    if "weights" in spec:
        model.load(spec.pop("weights"))
    max_det = spec.pop("max_det")
    model.train(name=name, **{**COMMON_ARGS, **spec})
    best = RTDETR(str(OUT / "ultralytics" / name / "weights" / "best.pt"))
    preds, ms = predict_ultralytics(best, val_frames, imgsz=COMMON_ARGS["imgsz"], max_det=max_det)
    save_preds(preds, OUT / f"preds_val_{name}.npz")
    metrics = evaluate(val_frames, preds)
    curves = read_curves(OUT / "ultralytics" / name)
    log_result(name, metrics, {"family": "query-based", "ms_per_img": round(ms, 1),
                               "train_min": round(curves["minutes"].iloc[-1], 1) if "minutes" in curves else None,
                               "config": str(spec | {"max_det": max_det})})
    print(name, metrics, f"{ms:.1f} ms/img")
    torch.cuda.empty_cache()

## Validation mAP over training: RT-DETR vs YOLOv8s with the same budget

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name in ["yolo_640", *RUNS]:
    run_dir = OUT / "ultralytics" / name
    if not (run_dir / "results.csv").exists():
        continue
    c = read_curves(run_dir)
    axes[0].plot(c.epoch, c.mAP, label=name)
    if "minutes" in c:
        axes[1].plot(c.minutes, c.mAP, label=name)
axes[0].set_xlabel("epoch")
axes[1].set_xlabel("GPU minutes")
for ax in axes:
    ax.set_ylabel("val mAP50-95")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures_detr_vs_yolo.png", dpi=120)
plt.show()

## Small objects and dense scenes
Val frames split into density groups by the number of GT objects; mAP per group.

In [ ]:
gt_count = val_frames.apply(lambda r: len(read_labels(r.label_path, r.width, r.height)[1]), axis=1)
val_frames["density"] = pd.cut(gt_count, [-1, 50, 150, 10_000], labels=["≤50", "51-150", ">150"])
rows = []
for name in ["yolo_640", *RUNS]:
    path = OUT / f"preds_val_{name}.npz"
    if not path.exists():
        continue
    p = load_preds(path)
    row = {"model": name, **{k: v for k, v in evaluate(val_frames, p).items() if k in ["mAP", "AP_small"]}}
    for d, frames_d in val_frames.groupby("density", observed=True):
        row[f"mAP {d} objs"] = evaluate(frames_d, p)["mAP"]
    gts, _ = error_breakdown(val_frames, p, score_thr=0.25)
    row.update({f"recall {k}": v for k, v in recall_by_size(gts).items()})
    rows.append(row)
print("frames per density group:", val_frames.density.value_counts().to_dict())
pd.DataFrame(rows)

TODO (your analysis): has RT-DETR converged within the budget (is the curve still rising)?
Effect of 300 vs 500 queries on dense frames; small-object behaviour vs YOLO; why bipartite
matching gives only one positive per GT (fewer, noisier gradients early on → slower convergence).